In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np 
import PIL.Image as Image
import pandas as pd
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms
import os
import matplotlib.pyplot as plt


c:\pc data\DATA SCIENCE AND ML\PYTHON\deep learning\Udemy - Complete Tensorflow 2 and Keras Deep Learning Bootcamp 2022-6\code place\envoo\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [6]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
class DatasetMaker(Dataset):
    def __init__(self,path,transform=None):
        super().__init__()
        self.map=pd.read_csv(path,index_col=0)
        self.transform=transform
    def __len__(self):
        return len(self.map)
    def __getitem__(self,idx):
        y=self.map['encoded_label'][idx]
        img=Image.open(self.map['ImagePath'][idx]).convert('L')
        if self.transform:
            img=self.transform(img)
        return img,y
        

In [3]:
#temp train data object just to calculate the mean and std that we will use in the main train data object and for the test and validation data too.
temp_train_transform=transforms.Compose([transforms.CenterCrop((480,480)),transforms.Resize((224,224)),transforms.ToTensor()])
train_path=r"DataSet\train_data.csv"
temp_train_Dataset=DatasetMaker(train_path,transform=temp_train_transform)
all=[temp_train_Dataset[i][0] for i in range(len(temp_train_Dataset))]
a=torch.stack(all,dim=0)
train_mean=a[:,0,:,:].mean()
train_std=a[:,0,:,:].std()

In [4]:
# we apply plausible (Physically) Augmentations to the train data only
# degrees=10 (rotation), translate=(0.05, 0.05) (horizontal/vertical shift)
# scale=(0.98, 1.02) (zoom in/out slightly)
train_transform=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.98, 1.02)),                        
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std)])

val_transform=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std)])


test_transform=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std)])

In [5]:
train_ds=DatasetMaker(path=r"DataSet\train_data.csv",transform=train_transform)
test_ds=DatasetMaker(path=r"DataSet\test_data.csv",transform=test_transform)
val_ds=DatasetMaker(path=r"DataSet\val_data.csv",transform=test_transform)

In [56]:
class CreateModel(nn.Module):
    def __init__(self,num_classes=2000):
        super().__init__()
        #input images are batch*1*224*224
        #first conv block
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)) # here we will have output of batch*64*112*112 max pool have default stride of kernel size which is 2 here.
        
        #second conv block
        self.block2 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)) # the ouput will be batch*128*56*56
        
        #3rd conv block 
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)) # the output will be batch*256*28*28
        
        # 4th conv block 
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)) # the output here will be batch*512*14*14
        
        #global average pooling layer to reduce the spatial dimensions to 1x1 so the dimension doesnt explode
        self.gap = nn.AdaptiveAvgPool2d((1, 1)) # the output will be batch*512*1*1 (it takes the average of each filter output)
        
        # fully connected block 
        self.fc1 = nn.Sequential(nn.Flatten(),
            nn.Dropout(p=0.5),
            nn.Linear(in_features=512*1*1,out_features=1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(in_features=1024,out_features=num_classes)) # this will output batch*2000 for the softmax function to use.      
        
        # define model behavior
    def forward(self,x):
        x=self.block1(x)
        x=self.block2(x)
        x=self.block3(x)
        x=self.block4(x)
        x=self.gap(x)
        x=self.fc1(x)
        return x

In [57]:
myCNN=CreateModel().to(device)

In [71]:
i[-1]

Parameter containing:
tensor([ 0.0140,  0.0202, -0.0145,  ...,  0.0262, -0.0081,  0.0168],
       device='cuda:0', requires_grad=True)

In [86]:
num_of_parameters=sum(p.numel() for p in myCNN.parameters())
print('the model parameters count is :',num_of_parameters,'\n ============================================ ')
for name,param in myCNN.named_parameters() :
    if param.requires_grad:
        print(name,f' |      num of paramaters: {param.numel()}')

the model parameters count is : 4127056 
block1.0.weight  |      num of paramaters: 576
block1.0.bias  |      num of paramaters: 64
block1.1.weight  |      num of paramaters: 64
block1.1.bias  |      num of paramaters: 64
block2.0.weight  |      num of paramaters: 73728
block2.0.bias  |      num of paramaters: 128
block2.1.weight  |      num of paramaters: 128
block2.1.bias  |      num of paramaters: 128
block3.0.weight  |      num of paramaters: 294912
block3.0.bias  |      num of paramaters: 256
block3.1.weight  |      num of paramaters: 256
block3.1.bias  |      num of paramaters: 256
block4.0.weight  |      num of paramaters: 1179648
block4.0.bias  |      num of paramaters: 512
block4.1.weight  |      num of paramaters: 512
block4.1.bias  |      num of paramaters: 512
fc1.2.weight  |      num of paramaters: 524288
fc1.2.bias  |      num of paramaters: 1024
fc1.5.weight  |      num of paramaters: 2048000
fc1.5.bias  |      num of paramaters: 2000


In [ ]:
train_dl=DataLoader(train_ds,batch_size=32,shuffle=True)
test_dl=DataLoader(test_ds,batch_size=32,shuffle=False)
val_dl=DataLoader(val_ds,batch_size=32,shuffle=False)
optimizer=torch.optim.Adam(myCNN.parameters(),lr=3e-4,weight_decay=1e-4)
loss=nn.CrossEntropyLoss()

In [ ]:
#function to evaluate model accuracy on a given dataloader
def evaluate(model, dataloader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            value, preds = torch.max(model(X), 1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total
# function to train the model for one epoch
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0 # Initialize to zero
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            _, preds = torch.max(outputs, 1)
            
            # Use size(0) to get the actual number of samples in the batch
            correct += (preds == y).sum().item()
            total += y.size(0) 
            
    return correct / total

def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs):
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(*epochs):
        model.train()
        running_loss = 0
        correct_train = 0
        total_train = 0
        
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = loss_fn(outputs, y)
            loss.backward()
            optimizer.step()
            # we track the trianing metrics manually this way instead of using the evaluate function to
            # avoid going over the trainiing data twice which is inefficient and time consuming.
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct_train += (preds == y).sum().item()
            total_train += y.size(0)

        # Calculate epoch metriics.
        epoch_loss = running_loss / len(train_loader)
        epoch_train_acc = correct_train / total_train
        epoch_val_acc = evaluate(model, val_loader)
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch [{i+60}/{80}] | Loss: {epoch_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Acc: {epoch_val_acc:.4f}")
        if (epoch+1)%5==0 :
            checkpoint = {
                'epoch': epoch+1, # The last completed epoch
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': epoch_loss, # Your last recorded loss
                'history': history # The training history containing loss and accuracy metrics
                }

            # Save to file
            torch.save(checkpoint, f"iris_model_final_v1_epoch{epoch+1}.pth")
            print(f"Checkpoint saved at epoch {epoch+1}")
        
    return history, model

**This cell has been run twice so we have a total of 40 epochs trained.**

In [96]:
# Train the model
history, trained_model = train_model(myCNN, train_dl, val_dl, optimizer, loss, epochs=range(20,40))

Epoch [21/range(20, 40)] | Loss: 4.4921 | Train Acc: 0.1001 | Val Acc: 0.1585
Epoch [22/range(20, 40)] | Loss: 4.3627 | Train Acc: 0.1095 | Val Acc: 0.2010
Epoch [23/range(20, 40)] | Loss: 4.2163 | Train Acc: 0.1246 | Val Acc: 0.2270
Epoch [24/range(20, 40)] | Loss: 4.1364 | Train Acc: 0.1324 | Val Acc: 0.2390
Epoch [25/range(20, 40)] | Loss: 4.0180 | Train Acc: 0.1484 | Val Acc: 0.2730
Epoch [26/range(20, 40)] | Loss: 3.8950 | Train Acc: 0.1611 | Val Acc: 0.3230
Epoch [27/range(20, 40)] | Loss: 3.8044 | Train Acc: 0.1728 | Val Acc: 0.3460
Epoch [28/range(20, 40)] | Loss: 3.7117 | Train Acc: 0.1814 | Val Acc: 0.3385
Epoch [29/range(20, 40)] | Loss: 3.6266 | Train Acc: 0.1964 | Val Acc: 0.3575
Epoch [30/range(20, 40)] | Loss: 3.5177 | Train Acc: 0.2066 | Val Acc: 0.3790
Epoch [31/range(20, 40)] | Loss: 3.4016 | Train Acc: 0.2286 | Val Acc: 0.4015
Epoch [32/range(20, 40)] | Loss: 3.3331 | Train Acc: 0.2371 | Val Acc: 0.3770
Epoch [33/range(20, 40)] | Loss: 3.2328 | Train Acc: 0.2515 | Va

**Now we will train for 20 more epochs to reach a total of 60 epochs.**

In [98]:
history, trained_model = train_model(myCNN, train_dl, val_dl, optimizer, loss, epochs=range(40,60))

Epoch [41/60] | Loss: 2.5731 | Train Acc: 0.3632 | Val Acc: 0.5980
Epoch [41/60] | Loss: 2.4920 | Train Acc: 0.3860 | Val Acc: 0.5050
Epoch [41/60] | Loss: 2.4196 | Train Acc: 0.3936 | Val Acc: 0.6145
Epoch [41/60] | Loss: 2.3772 | Train Acc: 0.4083 | Val Acc: 0.4915
Epoch [41/60] | Loss: 2.2794 | Train Acc: 0.4246 | Val Acc: 0.6505
Checkpoint saved at epoch 45
Epoch [41/60] | Loss: 2.2404 | Train Acc: 0.4333 | Val Acc: 0.6535
Epoch [41/60] | Loss: 2.1881 | Train Acc: 0.4461 | Val Acc: 0.6440
Epoch [41/60] | Loss: 2.1132 | Train Acc: 0.4594 | Val Acc: 0.6600
Epoch [41/60] | Loss: 2.0398 | Train Acc: 0.4718 | Val Acc: 0.7145
Epoch [41/60] | Loss: 1.9971 | Train Acc: 0.4788 | Val Acc: 0.7240
Checkpoint saved at epoch 50
Epoch [41/60] | Loss: 1.9578 | Train Acc: 0.4898 | Val Acc: 0.7385
Epoch [41/60] | Loss: 1.9102 | Train Acc: 0.5009 | Val Acc: 0.6415
Epoch [41/60] | Loss: 1.8343 | Train Acc: 0.5185 | Val Acc: 0.6625
Epoch [41/60] | Loss: 1.7998 | Train Acc: 0.5258 | Val Acc: 0.7305
Epoc

**since we hit the 60 epoch mark, its time for a drop in the learning rate from karpathy constant to 1e-4 and also add some more data augmantation**

In [100]:
# now we added more data augmantation such as color jitter, gaussian blur and random erasing to make the model more robust to different 
# lighting conditions and occlusions since in real world iris cameras can get out of focus and blurry and alsso the random erasing will help
# with the cases where the iris is hidden and occlused by the eyelids or the lashes 
# and also to add more variance to the data which should help the model generalize better.
train_transform_more_augmantaion=transforms.Compose([transforms.CenterCrop((480,480)),
                                    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.98, 1.02)),
                                    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
                                    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0)),                        
                                    transforms.Resize((224,224)),
                                    transforms.ToTensor(),
                                    transforms.Normalize(mean=train_mean,std=train_std),
                                    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3))])

train_ds_more_augmantaion=DatasetMaker(path=r"DataSet\train_data.csv",transform=train_transform_more_augmantaion)
train_dl_more_augmantaion=DataLoader(train_ds_more_augmantaion,batch_size=32,shuffle=True)